In [1]:
import sqlite3
import pandas as pd
import json

db_path = "part_lib/jlcpcb-components.sqlite3"

with sqlite3.connect(db_path) as conn:
    rows = pd.read_sql_query(
        """
        SELECT
            lcsc,
            category_id,
            mfr,
            package,
            joints,
            manufacturer_id,
            basic,
            description,
            datasheet,
            stock,
            price,
            last_update,
            extra
        FROM components
        LIMIT 5000
        """,
        conn,
    )

extra_series = rows["extra"].apply(lambda s: json.loads(s) if s else {})

# Pull attributes out intact
attributes_col = extra_series.apply(lambda d: d.get("attributes"))

# Drop attributes before flattening everything else
extra_df = pd.json_normalize(
    extra_series.apply(lambda d: {k: v for k, v in d.items() if k != "attributes"})
)

combined = pd.concat(
    [rows.drop(columns=["extra"]), extra_df, attributes_col.rename("attributes")],
    axis=1,
)

combined.head(3)


,lcsc,category_id,mfr,package,joints,manufacturer_id,basic,description,datasheet,stock,...,category.id1,category.id2,category.name1,category.name2,manufacturer.id,manufacturer.name,datasheet.pdf,quantity1,quantity3,attributes
0,1002,1,GZ1608D601TF,0603,2,1,1,,https://www.lcsc.com/datasheet/lcsc_datasheet_...,680254,...,10991.0,527.0,Filters,Ferrite Beads,270.0,Sunlord,https://wmsc.lcsc.com/wmsc/upload/file/pdf/v2/...,NaN,NaN,"{'DC Resistance': '450mΩ', 'Impedance @ Freque..."
1,1003,1,GZ1608D151TF,0603,2,1,0,,https://www.lcsc.com/datasheet/lcsc_datasheet_...,1870,...,10991.0,527.0,Filters,Ferrite Beads,270.0,Sunlord,https://wmsc.lcsc.com/wmsc/upload/file/pdf/v2/...,NaN,NaN,"{'DC Resistance': '-', 'Impedance @ Frequency'..."
2,1005,1,CBG160808U820T,0603,2,2,0,,,2178,...,10991.0,527.0,Filters,Ferrite Beads,63.0,FH (Guangdong Fenghua Advanced Tech),,NaN,NaN,"{'DC Resistance': '-', 'Impedance @ Frequency'..."
